# Parametric asset-only workbook generator

Creates an extended input workbook where all decision variables are defined directly in the **Assets** sheet and export/safety controls are exposed in **Settings**.

In [1]:
# Run this whole cell in Colab or Jupyter.

!pip -q install yfinance openpyxl pandas numpy

from pathlib import Path
import math
import numpy as np
import pandas as pd
import yfinance as yf

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.comments import Comment

# -------------------------
# Configuration
# -------------------------
TICKERS = [
    "NVDA", "AAPL", "MSFT", "AVGO", "MU", "ORCL",
    "AMD", "LRCX", "INTC", "CSCO", "AMAT", "KLAC"
]

# Each row in the final Assets sheet is one selectable option / one qubit.
# You can define any number of options per ticker.
# Example:
# OPTION_TARGETS_BY_TICKER = {
#     "NVDA": [100000, 150000],
#     "AAPL": [75000],
#     "MSFT": [50000, 100000, 150000],
#     "AVGO": [60000, 90000, 120000, 180000],
# }
OPTION_TARGETS_BY_TICKER = {ticker: [100000, 150000] for ticker in TICKERS}

OUTPUT_PATH = Path("parametric_assets_only_input_extended.xlsx")

NAME_MAP = {
    "NVDA": "NVIDIA",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AVGO": "Broadcom",
    "MU": "Micron Technology",
    "ORCL": "Oracle",
    "AMD": "Advanced Micro Devices",
    "LRCX": "Lam Research",
    "INTC": "Intel",
    "CSCO": "Cisco",
    "AMAT": "Applied Materials",
    "KLAC": "KLA",
}

SETTINGS_ROWS = [
    ("budget_usd", 1_000_000, "Target total budget"),
    ("risk_free_rate_annual", 0.04, "Annual risk-free rate used in excess-return reward"),
    ("lambda_budget", 50.0, "Budget deviation penalty"),
    ("lambda_variance", 6.0, "Variance contribution weight"),
    ("lambda_exclusive", 0.0, "Unused in current notebook; kept for transparency"),

    ("top_n_export", 20, "Number of candidate portfolios exported to overview sheets"),
    ("refresh_market_data", 0, "1 = refresh prices/returns/volatility from yfinance before solving"),

    ("enable_qaoa", 1, "1 = run QAOA, 0 = skip QAOA"),
    ("qaoa_p", 1, "QAOA depth / number of layers"),
    ("qaoa_maxiter", 60, "Optimizer iteration budget for QAOA"),
    ("qaoa_shots", 4096, "Shot count used when exact probability extraction is disabled"),
    ("qaoa_exact_probability_max_qubits", 20, "Use exact qml.probs only if n is at or below this threshold"),
    ("qaoa_max_qubits_allowed", 24, "Auto-skip QAOA if number of decision variables exceeds this threshold"),

    ("qaoa_export_mode", "top_k", "top_k or all_filtered"),
    ("qaoa_min_probability_to_export", 1e-12, "Exact-mode export ignores states below this probability"),
    ("qaoa_max_export_rows", 5000, "Maximum number of QAOA rows kept for export"),
    ("qaoa_export_feasible_only", 0, "1 = only export states within feasibility tolerance"),
    ("qaoa_feasibility_budget_tolerance_usd", 2500.0, "Abs budget gap tolerance used by feasible-only export"),
    ("qaoa_export_sort_by", "probability", "probability, qubo_value, sharpe_like, portfolio_return, abs_budget_gap"),

    ("enable_classical_search", 1, "1 = run classical local-search heuristic, 0 = skip it"),
    ("classical_max_qubits_allowed", 30, "Auto-skip classical heuristic if number of decision variables exceeds this threshold"),
    ("classical_random_search_samples", 8000, "Number of random classical starts"),
    ("classical_local_search_starts", 40, "Number of single-hot classical starts"),
    ("classical_max_neighbor_evals", 200000, "Safety cap on total 1-flip neighbor evaluations in greedy search"),

    ("overview_classical_pool", 300, "How many classical candidates feed the combined overview pool"),
    ("overview_qaoa_pool", 500, "How many QAOA candidates feed the combined overview pool"),
    ("result_candidate_limit_per_solver", 500, "How many rows to write to each raw solver output sheet"),
    ("rng_seed", 42, "Random seed for classical search and QAOA initialization"),
]

# -------------------------
# Download data
# -------------------------
prices = yf.download(
    tickers=TICKERS,
    period="12mo",
    interval="1d",
    auto_adjust=True,
    progress=False,
)["Close"]

if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.dropna(how="all").ffill().dropna()
rets = prices.pct_change().dropna()

total_return_12m = prices.iloc[-1] / prices.iloc[0] - 1
ann_vol = rets.std() * np.sqrt(252)
mean_daily = rets.mean()
std_daily = rets.std()
daily_cov = rets.cov()
annual_cov = daily_cov * 252
latest_price = prices.iloc[-1]

# -------------------------
# Build option rows
# -------------------------
option_rows = []
for ticker in TICKERS:
    current_price = float(latest_price[ticker])
    targets = OPTION_TARGETS_BY_TICKER.get(ticker, [])
    for idx, target_usd in enumerate(targets, start=1):
        shares = max(int(round(float(target_usd) / current_price)), 1)
        approx_cost = shares * current_price
        if abs(target_usd) >= 1000 and abs(target_usd) % 1000 == 0:
            label = f"{int(target_usd/1000)}k"
        else:
            label = f"opt{idx}"
        decision_id = f"{ticker}_opt{idx}"
        option_rows.append({
            "decision_id": decision_id,
            "Ticker": ticker,
            "Company": NAME_MAP.get(ticker, ticker),
            "Asset Group": ticker,
            "Option Label": label,
            "Shares": shares,
            "Approx Cost USD": approx_cost,
            "Expected Return Proxy": float(total_return_12m[ticker]),
            "Annual Volatility": float(ann_vol[ticker]),
            "Current Price (USD)": current_price,
            "Mean Daily Return": float(mean_daily[ticker]),
            "Std Daily Return": float(std_daily[ticker]),
            "Allowed": 1,
            "Price Source Status": "Imported via yfinance",
            "Source URL": "https://pypi.org/project/yfinance/",
        })

assets_df = pd.DataFrame(option_rows)

# -------------------------
# Workbook
# -------------------------
wb = Workbook()
dark = PatternFill("solid", fgColor="1F4E78")
section = PatternFill("solid", fgColor="D9EAF7")
white_bold = Font(color="FFFFFF", bold=True)
bold = Font(bold=True)

# ReadMe
ws = wb.active
ws.title = "ReadMe"
ws["A1"] = "Workbook overview"
ws["A1"].fill = dark
ws["A1"].font = white_bold
readme_lines = [
    "Each row in Assets is one selectable option and therefore one qubit.",
    "There is no separate blocks sheet anymore. Block structure is fully encoded by the rows in Assets.",
    "Repeat a ticker as often as needed to create multiple selectable options for one asset.",
    "Different assets can have different numbers of options.",
    "The processor uses Approx Cost USD directly.",
    "If Shares is filled and refresh_market_data=1, the processor can recompute Approx Cost USD from Shares × Current Price.",
    "Exact QAOA mode now supports filtered export controls so that large exact-state outputs do not blow up Excel writing.",
]
for i, line in enumerate(readme_lines, start=3):
    ws.cell(i, 1, line)
ws.column_dimensions["A"].width = 140

# Assets
ws = wb.create_sheet("Assets")
ws["A1"] = "Selectable option universe (one row = one binary decision variable)"
ws["A1"].fill = dark
ws["A1"].font = white_bold

asset_headers = list(assets_df.columns)
for c, h in enumerate(asset_headers, start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold

for r, row in enumerate(assets_df.itertuples(index=False), start=3):
    for c, value in enumerate(row, start=1):
        ws.cell(r, c, value)

for col in ["G", "J"]:
    for r in range(3, ws.max_row + 1):
        ws[f"{col}{r}"].number_format = '#,##0.00'
for col in ["H", "I", "K", "L"]:
    for r in range(3, ws.max_row + 1):
        ws[f"{col}{r}"].number_format = '0.0000'

asset_widths = [18, 10, 24, 12, 14, 10, 16, 18, 16, 16, 16, 16, 10, 22, 28]
for i, width in enumerate(asset_widths, start=1):
    ws.column_dimensions[chr(64+i)].width = width
ws.freeze_panes = "A3"
ws["A2"].comment = Comment("Unique identifier for each selectable option. Keep unique per row.", "OpenAI")
ws["B2"].comment = Comment("Repeat the same ticker on multiple rows to create multiple selectable options for one asset.", "OpenAI")
ws["D2"].comment = Comment("Optional grouping field. It is informational in the current processor.", "OpenAI")
ws["G2"].comment = Comment("Search uses Approx Cost USD directly. If Shares is filled and market refresh is enabled, the processor can recompute this value from Shares × Current Price.", "OpenAI")

# Settings
ws = wb.create_sheet("Settings")
ws["A1"] = "Model and solver settings"
ws["A1"].fill = dark
ws["A1"].font = white_bold
for c, h in enumerate(["Key", "Value", "Description"], start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold
for r, row in enumerate(SETTINGS_ROWS, start=3):
    for c, value in enumerate(row, start=1):
        ws.cell(r, c, value)
ws.column_dimensions["A"].width = 38
ws.column_dimensions["B"].width = 18
ws.column_dimensions["C"].width = 78
ws.freeze_panes = "A3"

# Returns
ws = wb.create_sheet("Returns")
ws["A1"] = "Per-ticker return and volatility inputs"
ws["A1"].fill = dark
ws["A1"].font = white_bold
return_headers = ["Ticker", "12M Return Proxy", "Annual Volatility", "Mean Daily Return", "Std Daily Return", "Source"]
for c, h in enumerate(return_headers, start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold

for r, ticker in enumerate(TICKERS, start=3):
    ws.cell(r, 1, ticker)
    ws.cell(r, 2, float(total_return_12m[ticker]))
    ws.cell(r, 3, float(ann_vol[ticker]))
    ws.cell(r, 4, float(mean_daily[ticker]))
    ws.cell(r, 5, float(std_daily[ticker]))
    ws.cell(r, 6, "https://pypi.org/project/yfinance/")

# Covariance
for title, matrix_df, sheet_name in [
    ("Daily return covariance matrix", daily_cov, "Covariance"),
    ("Annualized covariance matrix", annual_cov, "AnnualizedCovariance"),
]:
    ws = wb.create_sheet(sheet_name)
    ws["A1"] = title
    ws["A1"].fill = dark
    ws["A1"].font = white_bold

    tickers = list(matrix_df.index)
    for c, ticker in enumerate(tickers, start=2):
        cell = ws.cell(2, c, ticker)
        cell.fill = section
        cell.font = bold
    for r, ticker in enumerate(tickers, start=3):
        cell = ws.cell(r, 1, ticker)
        cell.fill = section
        cell.font = bold
        for c, ticker2 in enumerate(tickers, start=2):
            ws.cell(r, c, float(matrix_df.loc[ticker, ticker2]))

# Price history
ws = wb.create_sheet("PriceHistory")
ws["A1"] = "Adjusted close price history"
ws["A1"].fill = dark
ws["A1"].font = white_bold
ws.cell(2, 1, "Date").fill = section
ws.cell(2, 1).font = bold
for c, ticker in enumerate(TICKERS, start=2):
    ws.cell(2, c, ticker).fill = section
    ws.cell(2, c).font = bold
for r, dt in enumerate(prices.index, start=3):
    ws.cell(r, 1, dt.to_pydatetime())
    for c, ticker in enumerate(TICKERS, start=2):
        ws.cell(r, c, float(prices.loc[dt, ticker]))

# Sources
ws = wb.create_sheet("Sources")
ws["A1"] = "Sources"
ws["A1"].fill = dark
ws["A1"].font = white_bold
source_rows = [
    ("Historical market data", "https://pypi.org/project/yfinance/"),
    ("Universe style reference", "https://www.slickcharts.com/sp500"),
    ("Generated", "This workbook was generated by a Python script using yfinance."),
]
for r, row in enumerate(source_rows, start=2):
    ws.cell(r, 1, row[0])
    ws.cell(r, 2, row[1])

# Basic formatting
for sheet in wb.worksheets:
    for col_cells in sheet.columns:
        letter = col_cells[0].column_letter
        if sheet.column_dimensions[letter].width is None:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col_cells[: min(len(col_cells), 200)])
            sheet.column_dimensions[letter].width = min(max(max_len + 2, 10), 28)

wb.save(OUTPUT_PATH)
print(f"Saved workbook to {OUTPUT_PATH.resolve()}")



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Saved workbook to /Users/danielhug/code/qubit-lab/QAOA-Optimizer/Version 3/parametric_assets_only_input_extended.xlsx
